In [16]:
import ssl
import socket
import datetime
import hashlib

# ANSI цвета
RED = "\033[91m"
GREEN = "\033[92m"
YELLOW = "\033[93m"
CYAN = "\033[96m"
RESET = "\033[0m"

def get_fingerprint(cert_der):
    sha256 = hashlib.sha256(cert_der).hexdigest()
    return ":".join(sha256[i:i+2] for i in range(0, len(sha256), 2)).upper()

def parse_time(s):
    return datetime.datetime.strptime(s, "%b %d %H:%M:%S %Y %Z")

def check_validity(cert):
    not_before = parse_time(cert["notBefore"])
    not_after = parse_time(cert["notAfter"])
    now = datetime.datetime.utcnow()

    valid_now = not_before <= now <= not_after
    days_left = (not_after - now).days

    warnings = []
    if not valid_now:
        warnings.append(f"{RED}❌ Certificate NOT valid now!{RESET}")
    elif days_left < 30:
        warnings.append(f"{YELLOW}⚠️ Certificate expires in {days_left} days{RESET}")

    if cert["issuer"] == cert["subject"]:
        warnings.append(f"{YELLOW}⚠️ Self-signed certificate{RESET}")

    return valid_now, days_left, warnings

def fetch_cert(host, port=443):
    print(f"\n{CYAN}Connecting to {host}:{port} ...{RESET}\n")

    context = ssl.create_default_context()
    context.check_hostname = False
    context.verify_mode = ssl.CERT_NONE

    with socket.create_connection((host, port)) as sock:
        with context.wrap_socket(sock, server_hostname=host) as tls:

            # TLS info
            protocol = tls.version()
            cipher = tls.cipher()

            # DER → PEM → временно для декодирования
            cert_der = tls.getpeercert(binary_form=True)
            pem = ssl.DER_cert_to_PEM_cert(cert_der)

            import tempfile
            with tempfile.NamedTemporaryFile(delete=False, mode='w', suffix=".pem") as f:
                f.write(pem)
                temp_path = f.name

            cert = ssl._ssl._test_decode_cert(temp_path)
            fingerprint = get_fingerprint(cert_der)
            sans = [v for k, v in cert.get("subjectAltName", []) if k == "DNS"]

            # Вывод в красивом формате
            print(f"{CYAN}Subject:{RESET} {cert['subject']}")
            print(f"{CYAN}Issuer:{RESET} {cert['issuer']}")
            print(f"{CYAN}Valid From:{RESET} {cert['notBefore']}")
            print(f"{CYAN}Valid To:{RESET}   {cert['notAfter']}")
            print(f"{CYAN}SHA256 Fingerprint:{RESET} {fingerprint}")
            print(f"{CYAN}SANs:{RESET} {', '.join(sans) if sans else 'None'}")

            valid, days_left, warns = check_validity(cert)
            print(f"{CYAN}Currently valid:{RESET} {GREEN}Yes{RESET}" if valid else f"{CYAN}Currently valid:{RESET} {RED}No{RESET}")
            color_days = GREEN if days_left >= 30 else YELLOW
            print(f"{CYAN}Days until expiry:{RESET} {color_days}{days_left}{RESET}")

            # TLS info
            print(f"\n{CYAN}TLS Protocol:{RESET} {protocol}")
            print(f"{CYAN}Cipher used:{RESET} {cipher}\n")

            # Предупреждения
            if warns:
                print(f"{RED}Warnings / Issues:{RESET}")
                for w in warns:
                    print(" ", w)


In [17]:
fetch_cert("google.com")
fetch_cert("example.com", 443)



Connecting to google.com:443 ...

Subject: ((('commonName', '*.google.com'),),)
Issuer: ((('countryName', 'US'),), (('organizationName', 'Google Trust Services'),), (('commonName', 'WE2'),))
Valid From: Oct 27 08:33:51 2025 GMT
Valid To:   Jan 19 08:33:50 2026 GMT
SHA256 Fingerprint: 30:D1:CE:98:2F:EB:E2:4A:53:22:1C:BF:BB:4E:BC:2C:CA:E1:CE:CF:B7:9D:20:26:F9:55:2A:6B:ED:AD:DA:83
SANs: *.google.com, *.appengine.google.com, *.bdn.dev, *.origin-test.bdn.dev, *.cloud.google.com, *.crowdsource.google.com, *.datacompute.google.com, *.google.ca, *.google.cl, *.google.co.in, *.google.co.jp, *.google.co.uk, *.google.com.ar, *.google.com.au, *.google.com.br, *.google.com.co, *.google.com.mx, *.google.com.tr, *.google.com.vn, *.google.de, *.google.es, *.google.fr, *.google.hu, *.google.it, *.google.nl, *.google.pl, *.google.pt, *.googleapis.cn, *.googlevideo.com, *.gstatic.cn, *.gstatic-cn.com, googlecnapps.cn, *.googlecnapps.cn, googleapps-cn.com, *.googleapps-cn.com, gkecnapps.cn, *.gkecnapps.c